# Product ID 수집

MAP API를 통해 활성화된 상품 ID 리스트를 수집합니다.

## Parameters

In [ ]:
# Papermill parameters (DAG에서 전달받음)
env = "stg"
dt = "2024-08-07"  
version_date = "20240807"
map_api_base_url = "https://api.map-stg.sktelecom.com"
map_api_key = None
gcs_bucket_name = "air-airflow-stg"

## 1. 라이브러리 및 설정 로드

In [ ]:
import requests
import json
from datetime import datetime
from typing import List, Dict, Any
from google.cloud import storage

print(f"MAP API Base URL: {map_api_base_url}")
print(f"API Key: {map_api_key[:10]}..." if map_api_key else "No API Key")

## 2. API 연결 설정

In [ ]:
# API 헤더 설정
headers = {
    "Content-Type": "application/json",
    "x-apim-key": map_api_key
}

# SQL 쿼리 준비
query = "SELECT productCode FROM mc_intg_prod WHERE pmSyncYn='Y' AND llmUseYn='Y' LIMIT 500"
print(f"실행할 쿼리: {query}")

## 3. Product ID 리스트 요청

In [ ]:
# API 요청 데이터 준비
request_data = {
    "query": query
}

url = f"{map_api_base_url}/search/data-productcode-query"

print(f"요청 URL: {url}")
print(f"요청 데이터: {request_data}")

# API 호출
try:
    response = requests.post(
        url,
        headers=headers,
        json=request_data
    )
    
    print(f"응답 상태 코드: {response.status_code}")
    print(f"응답 헤더: {dict(response.headers)}")
    
    response.raise_for_status()
    
except requests.exceptions.RequestException as e:
    print(f"❌ API 요청 실패: {str(e)}")
    raise

## 4. 응답 데이터 처리

In [ ]:
# JSON 응답 파싱
response_data = response.json()
print(f"응답 데이터 구조: {list(response_data.keys())}")

# 데이터 추출
if 'data' in response_data:
    data_list = response_data['data']
    print(f"총 데이터 개수: {len(data_list)}")
    
    if data_list:
        print(f"첫 번째 데이터 샘플: {data_list[0]}")
    
    # Product ID 리스트 추출
    product_ids = []
    for item in data_list:
        if 'productCode' in item:
            product_ids.append(item['productCode'])
        else:
            print(f"⚠️ productCode 없는 항목: {item}")
    
    print(f"추출된 Product ID 개수: {len(product_ids)}")
    if product_ids:
        print(f"첫 5개 Product ID: {product_ids[:5]}")
        
else:
    print(f"❌ 응답에 'data' 키가 없습니다: {response_data}")
    raise ValueError("Invalid API response format")

## 5. 결과 저장 및 다음 단계로 전달

In [ ]:
# 결과 요약
result_summary = {
    "timestamp": datetime.now().isoformat(),
    "total_product_ids": len(product_ids),
    "api_response_status": response.status_code,
    "query_executed": query
}

print("\n=== 수집 결과 요약 ===")
for key, value in result_summary.items():
    print(f"{key}: {value}")

# 버전 날짜 설정 (파라미터로 받지 않은 경우 오늘 날짜 사용)
if not version_date:
    version_date = datetime.now().strftime("%Y%m%d")

print(f"\n버전 날짜: {version_date}")

# Product ID 리스트 데이터 준비
product_ids_data = {
    "collected_product_ids": product_ids,
    "collection_summary": result_summary,
    "version_date": version_date,
    "collection_date": dt
}

# GCS 경로 설정
gcs_path = f"product_id_list/{dt}/product_ids_{version_date}.json"

# GCS에 업로드
try:
    storage_client = storage.Client()
    bucket = storage_client.bucket(gcs_bucket_name)
    blob = bucket.blob(gcs_path)
    
    # JSON 데이터를 문자열로 변환 후 업로드
    json_data = json.dumps(product_ids_data, ensure_ascii=False, indent=2)
    blob.upload_from_string(json_data, content_type='application/json')
    
    gcs_full_path = f"gs://{gcs_bucket_name}/{gcs_path}"
    
    print(f"\n✅ Product ID 리스트 GCS 저장 완료: {gcs_full_path}")
    print(f"📊 저장된 Product ID 개수: {len(product_ids)}개")
    
except Exception as e:
    print(f"❌ GCS 저장 실패: {str(e)}")
    raise

# 출력 변수 (Papermill에서도 사용 가능하도록)
collected_product_ids = product_ids
collection_summary = result_summary

print(f"\n📤 다음 단계로 전달할 Product ID 개수: {len(collected_product_ids)}")

## 완료

Product ID 수집이 완료되었습니다. 다음 단계에서 각 Product ID에 대한 상세 정보를 수집합니다.